# Customer Support Routing Evaluation

Our evaluation of **Muse Spark Open on Fireworks**. Course materials supply reference tickets; human review, routing policy, failure analysis and measurements belong to this project.

**Current stage:** review labels and validate the integration. No baseline/improved results have been measured. All paid execution switches default to `False`.

In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src/support_eval").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/support_eval").exists(), "Open this notebook inside the project"
sys.path.insert(0, str(ROOT / "src"))
from support_eval.config import load_settings
from support_eval.dataset import freeze_dataset, load_frozen
from support_eval.agent import build_agent
from support_eval.evaluation import (make_run_spec, run_experiment, ensure_langsmith_dataset,
                                     score_results, compare_runs, RunSpec, write_json, cases_hash)
settings = load_settings(ROOT / ".env")
print("Model:", settings.model)
print("Credentials configured:", bool(settings.fireworks_api_key.get_secret_value()), bool(settings.langsmith_api_key.get_secret_value()))
print("Full-run integration gate:", settings.integration_verified)

Model: accounts/fireworks/models/muse-spark-open
Credentials configured: True True
Full-run integration gate: True


## 1. Review the tickets

Review `reports/label_review_guide.md` and `reports/label_policy.md`. Enter decisions in the CSV. A suggested label is not an approval. The original source labels remain intact; all corrections need a note. We freeze only after every included case has been reviewed.

In [2]:
review = pd.read_csv(ROOT / "data/review/label_review.csv", dtype=str, keep_default_na=False)
display(review.review_status.value_counts().rename("cases"))
display(review[["case_id", "ticket_text", "original_label", "reviewed_label", "review_status"]].head(12))

review_status
approved    100
Name: cases, dtype: int64

,case_id,ticket_text,original_label,reviewed_label,review_status
0,course-t000,"Hi, I ordered a blender 5 days ago and the tra...",order_status,order_status,approved
1,course-t001,"Hello, I placed an order for a blender five da...",order_status,order_status,approved
2,course-t002,"Hey, I ordered a blender about five days ago, ...",order_status,order_status,approved
3,course-t003,"Hi there, it’s been five days since I ordered ...",order_status,order_status,approved
4,course-t004,I never received my order and I want my money ...,order_status,refund_request,approved
5,course-t005,"I still haven't gotten my order, and I demand ...",order_status,refund_request,approved
6,course-t006,Can someone please help me? My order hasn't ar...,order_status,refund_request,approved
7,course-t007,"I can't believe this! My order is missing, and...",order_status,refund_request,approved
8,course-t008,Order shows delivered last Tuesday but it's no...,order_status,order_status,approved
9,course-t009,I see that my order was marked as delivered la...,order_status,order_status,approved


## 2. Freeze the reviewed dataset

Set `FREEZE_REVIEWED_DATA=True` only after review. Freezing is a local operation with no API usage. The folder cannot be overwritten. A changed dataset needs a new version and new runs.

In [3]:
FREEZE_REVIEWED_DATA = False
DATASET_VERSION = "v1"
FROZEN = ROOT / "data/frozen" / DATASET_VERSION
if FREEZE_REVIEWED_DATA:
    freeze_dataset(review, FROZEN, DATASET_VERSION)
if FROZEN.exists():
    cases, dataset_manifest = load_frozen(FROZEN)
    print("Frozen cases:", len(cases), "SHA-256:", dataset_manifest["sha256"])
else:
    cases = None
    print("Waiting for human review and dataset freezing.")

Frozen cases: 100 SHA-256: 6393654a0bbfb44b3d28dffac7f720c8e857f480dd1480fca4d27cbdbf0efe1d


## 3. Integration pilot — five paid calls when enabled

Before enabling: verify current Fireworks rates and that `max_tokens` limits total billable generation for Muse Spark Open. Confirm your LangSmith region/workspace. The pilot uses one reviewed case per category and writes a separate LangSmith dataset. It does not count as the full baseline.

Inspect each trace and usage afterward. Only set `INTEGRATION_VERIFIED=true` in `.env` once output schema, token accounting, cap and tracing have been verified. Do not change model/settings between the subsequent full runs.

In [4]:
def execute_saved_run(run_id, prompt_path, prompt_version, *, pilot=False):
    from langsmith import Client
    config = load_settings(ROOT / ".env")
    config.require_credentials()
    frozen_cases, manifest = load_frozen(FROZEN)
    selected = frozen_cases.groupby("expected_category", sort=True).head(1).copy() if pilot else frozen_cases
    if not pilot and not config.integration_verified:
        raise ValueError("Complete the integration pilot before full evaluation")
    prompt = (ROOT / prompt_path).read_text()
    version = manifest["version"] + ("-pilot" if pilot else "")
    selected_manifest = {**manifest, "version": version, "sha256": cases_hash(selected)}
    client = Client(api_key=config.langsmith_api_key.get_secret_value(), api_url=config.endpoint,
                    **({"workspace_id": config.workspace_id} if config.workspace_id else {}))
    dataset_id = ensure_langsmith_dataset(selected, selected_manifest, client)
    spec = make_run_spec(run_id, selected, version, prompt, config, prompt_version)
    spec.langsmith_dataset_id = dataset_id
    agent = build_agent(config, prompt)
    results = run_experiment(agent, selected, spec, config, ROOT / "artifacts", client=client, pilot=pilot)
    run_folder = ROOT / "artifacts/runs" / run_id
    stored_manifest = json.loads((run_folder / "manifest.json").read_text())
    if stored_manifest["status"] == "complete":
        metrics = score_results(results, spec)
        write_json(run_folder / "metrics.json", metrics)
        display(pd.Series(metrics).drop(["per_category", "confusion_matrix"]))
    else:
        print("Incomplete run; inspect saved attempts and usage before proceeding.")
    return results

RUN_PILOT = False
if RUN_PILOT:
    pilot_results = execute_saved_run("pilot-001", "prompts/baseline.txt", "baseline", pilot=True)
else:
    print("Pilot disabled; no paid calls.")

Pilot disabled; no paid calls.


## 4. Full baseline

Targets fixed before measurement: accuracy ≥95%, macro F1 ≥0.90, valid response rate 100%, p95 latency ≤15 seconds, mean inference cost ≤$0.02/ticket. Missing targets is a valid finding.

Use a new run ID for each deliberate experiment. The runner blocks reusing any existing ID. Do not reuse the course's published scores as ours.

In [5]:
RUN_BASELINE = False
if RUN_BASELINE:
    baseline_results = execute_saved_run("baseline-001", "prompts/baseline.txt", "baseline")
else:
    print("Baseline disabled; no paid calls.")

Baseline disabled; no paid calls.


## 5. Diagnose, then change one instruction

Review all baseline failures and a sample of passes. Record concrete cues, the wrong route and the expected boundary. Cluster actual failures into a few categories, rank frequency/business impact, and review the interpretation together.

Create `reports/failure_analysis.md` and `prompts/improved.txt` only after this analysis. Record the one change, expected benefit and possible regression. Do not copy the starter's improved prompt. If labels change, create a new frozen dataset and rerun both versions.

In [6]:
RUN_IMPROVED = False
if RUN_IMPROVED:
    improved_results = execute_saved_run("improved-001", "prompts/improved.txt", "improved")
else:
    print("Improved run disabled; no paid calls.")

Improved run disabled; no paid calls.


## 6. Replay the comparison — no model calls

This section loads existing complete runs. It verifies case identity, labels and model/settings before comparison. Costs with missing usage remain unknown. For incomplete runs, inspect individual records without reporting headline scores.

In [7]:
def read_complete_run(run_id):
    folder = ROOT / "artifacts/runs" / run_id
    manifest = json.loads((folder / "manifest.json").read_text())
    if manifest["status"] != "complete":
        raise ValueError("Incomplete run: cannot report headline comparison")
    frame = pd.read_json(folder / "results.jsonl", lines=True)
    spec = RunSpec.model_validate(manifest)
    score_results(frame, spec)  # validates full coverage and identity
    return frame, spec

base_dir = ROOT / "artifacts/runs/baseline-001"
improved_dir = ROOT / "artifacts/runs/improved-001"
if base_dir.exists() and improved_dir.exists():
    baseline, baseline_spec = read_complete_run("baseline-001")
    improved, improved_spec = read_complete_run("improved-001")
    paired = compare_runs(baseline, improved, baseline_spec, improved_spec)
    display(paired.outcome.value_counts())
    display(pd.DataFrame({"baseline": score_results(baseline, baseline_spec),
                          "improved": score_results(improved, improved_spec)}).loc[
        ["accuracy", "macro_f1", "valid_response_rate", "p95_latency_seconds", "mean_cost_usd"]])
    display(paired[paired.outcome.isin(["fixed", "regression"])])
else:
    print("No paired measured runs yet. Source results are not substituted.")

No paired measured runs yet. Source results are not substituted.


## 7. Submission checklist

- Completed workbook with our labels, annotations, failure categories and paired results.
- Executed notebook, frozen cases, prompts and saved experiment artifacts.
- LangSmith experiment links and an inspected ticket trace.
- Report covering deltas, regressions, cost/latency, synthetic-data limits and same-set evaluation.
- Ramesh's narrated demo using `reports/demo_outline.md`.

**Scope:** one focused improvement on the reviewed course corpus. Optional judge work is not attempted. Disclose the difference from the general handout's 3–4 changes / 30–50 cases. Do not claim positive improvement or production readiness without evidence.